In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *

In [0]:
df_user = spark.readStream.format("cloudFiles")\
                .option("cloudFiles.format", "parquet")\
                .option("cloudFiles.schemaLocation", "abfss://silver@spotifyazurestoragesink.dfs.core.windows.net/DimUser/checkpoint")\
                .option("schemaEvolutionMode", "addNewColumns")\
                .option("cloudFiles.includeExistingFiles", "true")\
                .load("abfss://bronze@spotifyazurestoragesink.dfs.core.windows.net/DimUser")

In [0]:
print(df_user.isStreaming)

In [0]:
query = (df_user.writeStream.format("memory")\
                    .queryName("user_stream")\
                    .outputMode("append")\
                    .option("checkpointLocation", "abfss://silver@spotifyazurestoragesink.dfs.core.windows.net/DimUser/checkpoint1")\
                    .trigger(once=True)\
                    .start())
query.awaitTermination()
display(spark.sql("select * from user_stream"))